In [ ]:
"""
We split our data into different segments and feed them to the model separately so it can learn and subsequently update its parameters.

During training, the dataset is divided into smaller **mini-batches**, and each batch is passed through the neural network separately.
After a batch goes through a layer, **Batch Normalization** can normalize the layer’s activations before they are passed to the next layer.
It calculates the mean and variance of the activations within that batch, normalizes them, and then uses learnable parameters to scale and shift the normalized values.
After the forward pass, the loss is calculated, backpropagation computes the gradients, and the model’s weights are updated before processing the next batch.
"""

In [8]:
# Implementation using TensorFlow

import tensorflow as tf

In [10]:
# Load and Preprocess the Dataset

# if have not downloaded dataset uncomment and use this
# (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# # Normalize pixel values
# x_train = x_train.astype("float32") / 255.0
# x_test = x_test.astype("float32") / 255.0

# # Flatten each 28×28 image into a 784-dimensional vector
# x_train = x_train.reshape(-1, 784)
# x_test = x_test.reshape(-1, 784)

# # Use a small subset for faster execution
# x_train = x_train[:5000]
# y_train = y_train[:5000]

import numpy as np
import struct
import os

def load_mnist_images(filename):
    with open(filename, 'rb') as f:
        magic, num, rows, cols = struct.unpack('>IIII', f.read(16))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data.reshape(num, rows, cols)

def load_mnist_labels(filename):
    with open(filename, 'rb') as f:
        magic, num = struct.unpack('>II', f.read(8))
        data = np.frombuffer(f.read(), dtype=np.uint8)
        return data


data_dir = os.path.join('data', 'MNIST', 'raw')

x_train = load_mnist_images(os.path.join(data_dir, 'train-images-idx3-ubyte'))
y_train = load_mnist_labels(os.path.join(data_dir, 'train-labels-idx1-ubyte'))
x_test  = load_mnist_images(os.path.join(data_dir, 't10k-images-idx3-ubyte'))
y_test  = load_mnist_labels(os.path.join(data_dir, 't10k-labels-idx1-ubyte'))

# Normalize pixel values
x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

# Flatten each 28×28 image into a 784-dimensional vector
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

# Use a small subset for faster execution
x_train = x_train[:5000]
y_train = y_train[:5000]

In [11]:
# Create the Neural Network with Batch Normalization

model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(64),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Activation("relu"),
    tf.keras.layers.Dense(10, activation="softmax")
])

In [13]:
#  Compile the Model

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [14]:
# Train the Model

model.fit(
    x_train,
    y_train,
    epochs=2,
    batch_size=32
)


Epoch 1/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7922 - loss: 0.7821
Epoch 2/2
157/157 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9180 - loss: 0.3499


In [1]:
# Implementing using PyTorch

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

In [2]:
# Load and Prepare the Dataset

# Convert images to tensors
transform = transforms.ToTensor()

# Load the MNIST training dataset
train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Use only the first 5000 samples
train_dataset = Subset(train_dataset, range(5000))

# Create a DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

In [3]:
# Define the Neural Network with Batch Normalization

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 64)
        self.bn = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


In [4]:
# Create the Model and Define the Loss Function and Optimizer

model = Model()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


In [5]:
# Train the Model

for epoch in range(2):
    running_loss = 0.0

    for images, labels in train_loader:

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")


Epoch 1, Loss: 0.8432
Epoch 2, Loss: 0.3665
